In [ ]:
import os
import glob
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import librosa
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import timm

import warnings
warnings.filterwarnings("ignore")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device=torch.device("cpu")
print(device)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# del cnn_model
# del crnn_model
# del effnet_model
# del effnet_aug_model

import gc

gc.collect()

torch.cuda.empty_cache()

In [ ]:
DATA_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"


STEMS_DIR = f"{DATA_DIR}/genres_stems"
MASHUPS_DIR = f"{DATA_DIR}/mashups"
ESC50_DIR = f"{DATA_DIR}/ESC-50-master/audio"

TEST_CSV = f"{DATA_DIR}/test.csv"
SUBMISSION_CSV = f"{DATA_DIR}/sample_submission.csv"

In [ ]:
GENRES = sorted(os.listdir(STEMS_DIR))
GENRE_TO_IDX = {g:i for i,g in enumerate(GENRES)}
IDX_TO_GENRE = {i:g for g,i in GENRE_TO_IDX.items()}
EPOCHS=40

print(GENRES)

In [ ]:
records = []

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)

    for song in os.listdir(genre_path):

        song_path = os.path.join(genre_path, song)

        stems = {
            "bass": os.path.join(song_path,"bass.wav"),
            "drums": os.path.join(song_path,"drums.wav"),
            "other": os.path.join(song_path,"other.wav"),
            "vocals": os.path.join(song_path,"vocals.wav")
        }

        records.append({
            "genre":genre,
            "genre_id":GENRE_TO_IDX[genre],
            "stems":stems
        })

df = pd.DataFrame(records)
df.head()

In [ ]:
noise_files = []

for f in os.listdir(ESC50_DIR):
    if f.endswith(".wav"):
        noise_files.append(os.path.join(ESC50_DIR,f))

print("Noise files:",len(noise_files))

In [ ]:
def load_audio(path, sr=22050):

    y, _ = librosa.load(path, sr=sr)

    return y

In [ ]:
genre_groups = {g: df[df.genre == g] for g in GENRES}

def synthetic_mashup(genre):

    main_song = genre_groups[genre].sample(1).iloc[0]

    bass = load_audio(main_song.stems["bass"])
    drums = load_audio(main_song.stems["drums"])

    # inject cross genre contamination
    other_genre = random.choice([g for g in GENRES if g != genre])
    other_song = genre_groups[other_genre].sample(1).iloc[0]

    vocals = load_audio(other_song.stems["vocals"])
    other = load_audio(other_song.stems["other"])

    min_len = min(len(bass), len(drums), len(vocals), len(other))

    bass = bass[:min_len]
    drums = drums[:min_len]
    vocals = vocals[:min_len]
    other = other[:min_len]

    w = np.random.uniform(0.4,1.4,4)

    mix = (
        w[0]*bass +
        w[1]*drums +
        0.6*w[2]*vocals +
        0.5*w[3]*other
    )

    mix = mix/(np.max(np.abs(mix))+1e-6)

    return mix

In [ ]:
def add_noise(signal, snr_db=10):

    noise_path = random.choice(noise_files)
    noise = load_audio(noise_path)

    if len(noise) < len(signal):
        noise = np.tile(noise, len(signal)//len(noise)+1)

    noise = noise[:len(signal)]

    signal_power = np.mean(signal**2)
    noise_power = np.mean(noise**2)

    factor = np.sqrt(signal_power/(10**(snr_db/10)*noise_power))

    noisy = signal + factor*noise

    return noisy

In [ ]:
def augment_audio(y, target_len=None):

    if random.random() < 0.5:
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y, rate=rate)

    if random.random() < 0.5:
        y = add_noise(y, snr_db=random.randint(5, 20))

    # ensure consistent length
    if target_len is not None:

        if len(y) > target_len:
            y = y[:target_len]

        else:
            pad = target_len - len(y)
            y = np.pad(y, (0, pad))

    return y

In [ ]:
def random_crop(y, crop_size):

    if len(y) <= crop_size:
        return np.pad(y, (0, crop_size - len(y)))

    start = random.randint(0, len(y) - crop_size)

    return y[start:start + crop_size]

In [ ]:
class StemDataset(Dataset):

    def __init__(self, df, sr=22050):

        self.df = df
        self.sr = sr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        y = synthetic_mashup(row.genre)

        y = random_crop(y, 22050 * 5)
        y = augment_audio(y, 22050 * 6)

        y = torch.tensor(y).float().unsqueeze(0)

        label = row.genre_id

        return y, label

In [ ]:
def collate_fn(batch):

    waves = [b[0] for b in batch]
    labels = [b[1] for b in batch]

    max_len = max([w.shape[1] for w in waves])

    padded = []

    for w in waves:
        pad = max_len - w.shape[1]
        padded.append(nn.functional.pad(w,(0,pad)))

    return torch.stack(padded), torch.tensor(labels)

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df.genre
)

In [ ]:
train_ds = StemDataset(train_df)
val_ds = StemDataset(val_df)

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
    # num_workers=2,
    # pin_memory=True
    num_workers=0,
    pin_memory=False
)
val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
     # num_workers=2,
    # pin_memory=True
    num_workers=0,
    pin_memory=False
)

In [ ]:
# mel_transform = torchaudio.transforms.MelSpectrogram(
#     sample_rate=22050,
#     n_fft=2048,
#     hop_length=128,
#     n_mels=256,
#     f_min=20,
#     f_max=11025
# ).to(device)
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=22050,
    n_fft=1024,
    hop_length=256,
    n_mels=128,
).to(device)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def extract_mfcc_from_stems(row):

    y = synthetic_mashup(row.genre)
    y = random_crop(y, 22050 * 6)

    mfcc = librosa.feature.mfcc(y=y, sr=22050, n_mfcc=40)

    return mfcc.mean(axis=1)

print("Extracting MFCC features...")

X = []
y = []

for i,row in tqdm(df.iterrows(), total=len(df)):
    
    feat = extract_mfcc_from_stems(row)
    
    X.append(feat)
    y.append(row.genre_id)

X = np.array(X)
y = np.array(y)

X_train,X_val,y_train,y_val = train_test_split(
    X,y,test_size=0.1,stratify=y
)

clf = LogisticRegression(max_iter=2000)

clf.fit(X_train,y_train)

pred = clf.predict(X_val)

print("Logistic Regression Macro F1:", f1_score(y_val,pred,average="macro"))

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.net = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128,len(GENRES))

    def forward(self,x):

        x = self.mel(x)
        x = torch.log(x + 1e-6)

        x = self.net(x)

        x = x.view(x.size(0),-1)

        return self.fc(x)

In [ ]:
# cnn_model = SimpleCNN().to(device)

# optimizer = torch.optim.Adam(cnn_model.parameters(),lr=1e-3)

# criterion = nn.CrossEntropyLoss()

# for epoch in range(5):

#     cnn_model.train()

#     total_loss = 0

#     for x,y in tqdm(train_loader):

#         x,y = x.to(device),y.to(device)

#         optimizer.zero_grad()

#         preds = cnn_model(x)

#         loss = criterion(preds,y)

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     print("Epoch",epoch,"Loss",total_loss/len(train_loader))

In [ ]:
class CRNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.cnn = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256,len(GENRES))

    def forward(self,x):

        x = self.mel(x)

        x = torch.log(x+1e-6)

        x = self.cnn(x)

        x = x.mean(dim=2)

        x = x.permute(0,2,1)

        x,_ = self.gru(x)

        x = x.mean(dim=1)

        return self.fc(x)

In [ ]:
# crnn_model = CRNN().to(device)

# optimizer = torch.optim.Adam(crnn_model.parameters(),lr=3e-4)

# criterion = nn.CrossEntropyLoss()

# for epoch in range(7):

#     crnn_model.train()

#     total_loss = 0

#     for x,y in tqdm(train_loader):

#         x,y = x.to(device),y.to(device)

#         optimizer.zero_grad()

#         preds = crnn_model(x)

#         loss = criterion(preds,y)

#         loss.backward()

#         optimizer.step()

#         total_loss += loss.item()

#     print("Epoch",epoch,"Loss",total_loss/len(train_loader))

In [ ]:
class EfficientNetAudio(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.freq_mask = torchaudio.transforms.FrequencyMasking(24)
        self.time_mask = torchaudio.transforms.TimeMasking(40)

        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    def forward(self,x):

        x = self.mel(x)

        x = torch.log(x+1e-6)

        x = (x - x.mean(dim=(2,3),keepdim=True)) / (x.std(dim=(2,3),keepdim=True)+1e-6)

        if self.training:
            x = self.freq_mask(x)
            x = self.time_mask(x)

        return self.backbone(x)

In [ ]:
effnet_model = EfficientNetAudio().to(device)

optimizer = torch.optim.AdamW(effnet_model.parameters(),lr=3e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)

criterion = nn.CrossEntropyLoss()

for epoch in range(25):

    effnet_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x,y = x.to(device),y.to(device)

        optimizer.zero_grad()

        preds = effnet_model(x)

        loss = criterion(preds,y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

In [ ]:
class AudioClassifier(nn.Module):

    def __init__(self):

        super().__init__()

        self.mel = mel_transform

        self.freq_mask = torchaudio.transforms.FrequencyMasking(48)
        self.time_mask = torchaudio.transforms.TimeMasking(120)

        self.backbone = timm.create_model(
            "tf_efficientnet_b0",
            pretrained=True,
            in_chans=1,
            num_classes=len(GENRES)
        )

    def forward(self,x):

        x = self.mel(x)
        x = torch.log(x + 1e-6)

        x = (x - x.mean(dim=(2,3), keepdim=True)) / (x.std(dim=(2,3), keepdim=True) + 1e-6)

        if self.training:
            if random.random() < 0.5:
                x = self.freq_mask(x)
            if random.random() < 0.5:
                x = self.time_mask(x)

        return self.backbone(x)

In [ ]:
effnet_aug_model = AudioClassifier().to(device)

optimizer = torch.optim.AdamW(effnet_aug_model.parameters(), lr=3e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
for epoch in range(EPOCHS):

    effnet_aug_model.train()

    total_loss = 0

    for x,y in tqdm(train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        lam = np.random.beta(0.4, 0.4)

        index = torch.randperm(x.size(0)).to(device)
        
        mixed_x = lam * x + (1 - lam) * x[index]
        
        y_a, y_b = y, y[index]
        
        preds = effnet_aug_model(mixed_x)
        
        loss = lam * criterion(preds, y_a) + (1 - lam) * criterion(preds, y_b)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(effnet_aug_model.parameters(), 1.0)
        
        optimizer.step()

        total_loss += loss.item()
    scheduler.step()

    print("Epoch",epoch,"Loss",total_loss/len(train_loader))

In [ ]:
def evaluate():

    effnet_aug_model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x,y in val_loader:

            x = x.to(device)
            y = y.to(device)

            preds = effnet_aug_model(x)

            preds = preds.argmax(1)

            correct += (preds==y).sum().item()
            total += len(y)

    return correct/total

print("Validation Accuracy:",evaluate())

In [ ]:
class MashupDataset(Dataset):

    def __init__(self,test_df):

        self.df = test_df

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):

        row = self.df.iloc[idx]

        path = os.path.join(DATA_DIR,row.filename)

        y,_ = librosa.load(path,sr=22050)

        y = torch.tensor(y).float().unsqueeze(0)

        return y,row.id

In [ ]:
test_df = pd.read_csv(TEST_CSV)

test_ds = MashupDataset(test_df)

test_loader = DataLoader(
    test_ds,
    batch_size=16,
    collate_fn=lambda b: collate_fn([(x,0) for x,_ in b])
)

In [ ]:
def predict_audio(audio):

    crop_size = 22050 * 6

    crops = []

    for start in range(0, len(audio) - crop_size, crop_size // 4):
        crops.append(audio[start:start+crop_size])

    if len(crops) == 0:
        crops.append(random_crop(audio,crop_size))

    preds = []

    for crop in crops:

        x = torch.tensor(crop).float().unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():

            # p1 = torch.softmax(cnn_model(x),1)
            # p2 = torch.softmax(crnn_model(x),1)
            p3 = torch.softmax(effnet_model(x),1)
            p4 = torch.softmax(effnet_aug_model(x),1)

            p = (
                # 0.15*p1 +  0.25*p2 + 0.35*p3 + 0.25*p4
                0.58 *p3 + 0.42*p4
            )

        preds.append(p.cpu().numpy())

    preds = np.mean(preds,axis=0)

    return preds.argmax()

In [ ]:
# cnn_model.eval()
# crnn_model.eval()
effnet_model.eval()
effnet_aug_model.eval()

predictions = []

for i,row in tqdm(test_df.iterrows(), total=len(test_df)):

    path = os.path.join(DATA_DIR, row.filename)

    y,_ = librosa.load(path, sr=22050)

    pred = predict_audio(y)

    predictions.append(pred)

In [ ]:
submission = pd.read_csv(SUBMISSION_CSV)

submission["genre"] = [IDX_TO_GENRE[p] for p in predictions]

submission.head()

In [ ]:
# del cnn_model
# del crnn_model
del effnet_model
del effnet_aug_model
torch.cuda.empty_cache()

In [ ]:
submission.to_csv("submission.csv",index=False)